# CLIP Embeddings (CPU)

Считает эмбеддинги объявлений из `image_paths` (до 10 фото) и сохраняет в `data/processed`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
import open_clip

PROJECT_ROOT = Path('/Users/zhasik/Desktop/krisha')
PARQUET_PATH = PROJECT_ROOT / 'data/index/index.parquet'
OUT_DIR = PROJECT_ROOT / 'data/processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)
MAX_IMAGES = 10

df = pd.read_parquet(PARQUET_PATH)
df['ad_id'] = df['ad_id'].astype(str)
print('rows:', len(df))


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_name = 'ViT-B-32'
pretrained = 'laion2b_s34b_b79k'
model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=pretrained, device=device)
model.eval()
print('device:', device)

def to_paths(v):
    if isinstance(v, list):
        return [Path(x) for x in v if str(x).strip()]
    return []

def encode_ad(paths):
    imgs = []
    for p in paths[:MAX_IMAGES]:
        try:
            img = Image.open(p).convert('RGB')
            imgs.append(preprocess(img))
        except Exception:
            continue
    if not imgs:
        return None
    x = torch.stack(imgs, dim=0).to(device)
    with torch.no_grad():
        z = model.encode_image(x)
        z = F.normalize(z, dim=-1)
    e = z.mean(dim=0).cpu().numpy().astype('float32')
    e = e / (np.linalg.norm(e) + 1e-12)
    return e


In [ ]:
ad_ids = []
embs = []
for _, r in df.iterrows():
    paths = to_paths(r.get('image_paths'))
    e = encode_ad(paths)
    if e is None:
        continue
    ad_ids.append(str(r['ad_id']))
    embs.append(e)

ad_ids = np.asarray(ad_ids, dtype='U')
ad_embs = np.asarray(embs, dtype='float32')
np.save(OUT_DIR / 'clip_vitb32_ad_ids.npy', ad_ids)
np.save(OUT_DIR / 'clip_vitb32_ad_emb.npy', ad_embs)
print('saved ids:', OUT_DIR / 'clip_vitb32_ad_ids.npy', ad_ids.shape)
print('saved emb:', OUT_DIR / 'clip_vitb32_ad_emb.npy', ad_embs.shape)
